# IoT device vulnerabilities  (MODEL SOLUTION, instructor copy, do not distribute)

# Coding problem, 4.1.c: Audit an IoT device for its dominant vulnerability

**Skill:** Analyze Risk · **Maps to:** 4.1.A, 4.1.C · **Language:** Python 3 (pytest)

## Problem
A security analyst is auditing smart devices on a network. Each device is described by a
dictionary of attributes. Build the classifier that inspects a device and returns its single most
severe vulnerability category, or `"secure"` if none of the checked issues are present.

A device can technically have more than one issue at once; when it does, report only the most
severe one, using this fixed priority order (highest severity first):

1. `"default_credentials"`, the device still uses its factory default password
2. `"outdated_firmware"`, the device's firmware is out of date
3. `"exposed_port"`, the device has an insecure management port open (23 = Telnet, 21 = FTP)
4. `"unauthenticated_api"`, the device's API accepts requests with no authentication
5. `"unencrypted_traffic"`, the device's network traffic is not encrypted
6. `"secure"`, none of the above apply

## Function contract
Implement `solve(device: dict) -> str` in the code editor (see the Reference section below).
Return the single verdict label string.

## Input
A dictionary with the following keys:
- `default_password` (bool), True if the factory default password is still in use
- `firmware_outdated` (bool), True if the firmware is out of date
- `open_ports` (list[int]), list of open network ports on the device
- `unauthenticated_api` (bool), True if the device's API requires no authentication
- `encrypted_traffic` (bool), True if the device's traffic is encrypted (note: False means
  traffic is UNencrypted, which is the risk condition)

An insecure management port is any of `23` (Telnet) or `21` (FTP) appearing in `open_ports`.

## Output
One of: `"default_credentials"`, `"outdated_firmware"`, `"exposed_port"`,
`"unauthenticated_api"`, `"unencrypted_traffic"`, `"secure"`.

## Examples
```
{"default_password": True, "firmware_outdated": True, "open_ports": [23], "unauthenticated_api": True, "encrypted_traffic": False}  ->  "default_credentials"
{"default_password": False, "firmware_outdated": True, "open_ports": [443], "unauthenticated_api": False, "encrypted_traffic": True}  ->  "outdated_firmware"
{"default_password": False, "firmware_outdated": False, "open_ports": [23, 443], "unauthenticated_api": False, "encrypted_traffic": True}  ->  "exposed_port"
{"default_password": False, "firmware_outdated": False, "open_ports": [443], "unauthenticated_api": True, "encrypted_traffic": True}  ->  "unauthenticated_api"
{"default_password": False, "firmware_outdated": False, "open_ports": [443], "unauthenticated_api": False, "encrypted_traffic": False}  ->  "unencrypted_traffic"
{"default_password": False, "firmware_outdated": False, "open_ports": [443], "unauthenticated_api": False, "encrypted_traffic": True}  ->  "secure"
```

## Constraints
- Deterministic, offline, standard library only.
- Priority order is fixed as listed above and must be applied exactly.
- Only ports `23` and `21` count as insecure/exposed management ports.
- Exactly one label returned (string, not list).
- Must pass all cases in the tests (visible + hidden).

## Reference
Any implementation that satisfies this interface can be swapped in (reference, student, or
composite) and must pass the tests unchanged.

### Entry point
```python
def solve(device: dict) -> str
```

### Input (`device`)
A dictionary with keys:
- `default_password` (bool), True if the factory default password is still in use.
- `firmware_outdated` (bool), True if the firmware is out of date.
- `open_ports` (list[int]), open network ports on the device.
- `unauthenticated_api` (bool), True if the device's API requires no authentication.
- `encrypted_traffic` (bool), True if the device's traffic is encrypted (False = the risk
  condition: traffic is unencrypted).

Insecure/exposed management ports: `{23, 21}` (Telnet, FTP). Any other port does not count as
"exposed" for this problem.

### Output (`result`)
Exactly one string from:
`{"default_credentials", "outdated_firmware", "exposed_port", "unauthenticated_api",
"unencrypted_traffic", "secure"}`

### Priority order (highest severity first)
When a device matches more than one condition, return only the highest-priority match:
1. `default_password is True` -> `"default_credentials"`
2. `firmware_outdated is True` -> `"outdated_firmware"`
3. any of `{23, 21}` present in `open_ports` -> `"exposed_port"`
4. `unauthenticated_api is True` -> `"unauthenticated_api"`
5. `encrypted_traffic is False` -> `"unencrypted_traffic"`
6. none of the above -> `"secure"`

### Guarantees / rules
- Pure function, deterministic, no I/O, no network. Standard library only.
- `open_ports` may be empty or contain any integers; only `23` and `21` trigger `"exposed_port"`.
- Missing keys should be treated as the "safe" default for that key (`False` for booleans,
  `[]` for `open_ports`, and `True` for `encrypted_traffic`, i.e. "assume secure/encrypted if
  not specified").
- Returns a single string (not a list).

### Acceptance
Your code passes when every test below shows a checkmark (some are hidden until you run them).


In [ ]:
# Reference solution
"""Reference solution for 4.1.c. MUST pass 100% of the tests.
Also the default 'blackbox' implementation students can swap out (see the hidden grading tests)."""

INSECURE_PORTS = {23, 21}


def solve(device: dict) -> str:
    """Audit an IoT device dict and return its single dominant vulnerability verdict.

    Args:
        device: dict with keys default_password (bool), firmware_outdated (bool),
            open_ports (list[int]), unauthenticated_api (bool), encrypted_traffic (bool).
            Missing keys default to the "safe" value for that key.

    Returns:
        One of: "default_credentials", "outdated_firmware", "exposed_port",
        "unauthenticated_api", "unencrypted_traffic", "secure".
    """
    if not isinstance(device, dict):
        return "secure"

    default_password = bool(device.get("default_password", False))
    firmware_outdated = bool(device.get("firmware_outdated", False))
    open_ports = device.get("open_ports", []) or []
    unauthenticated_api = bool(device.get("unauthenticated_api", False))
    encrypted_traffic = bool(device.get("encrypted_traffic", True))

    # Priority order: default_credentials > outdated_firmware > exposed_port >
    # unauthenticated_api > unencrypted_traffic > secure

    if default_password:
        return "default_credentials"

    if firmware_outdated:
        return "outdated_firmware"

    if any(port in INSECURE_PORTS for port in open_ports):
        return "exposed_port"

    if unauthenticated_api:
        return "unauthenticated_api"

    if not encrypted_traffic:
        return "unencrypted_traffic"

    return "secure"


## Self-check tests
Run the cell below to check your `solve` implementation above.

In [ ]:
import __main__ as impl  # the functions defined above

# ---- visible cases -------------------------------------------------------
def test_example_default_credentials():
    device = {"default_password": True, "firmware_outdated": True, "open_ports": [23],
              "unauthenticated_api": True, "encrypted_traffic": False}
    assert impl.solve(device) == "default_credentials"


def test_example_outdated_firmware():
    device = {"default_password": False, "firmware_outdated": True, "open_ports": [443],
              "unauthenticated_api": False, "encrypted_traffic": True}
    assert impl.solve(device) == "outdated_firmware"


def test_example_exposed_port():
    device = {"default_password": False, "firmware_outdated": False, "open_ports": [23, 443],
              "unauthenticated_api": False, "encrypted_traffic": True}
    assert impl.solve(device) == "exposed_port"


def test_example_unauthenticated_api():
    device = {"default_password": False, "firmware_outdated": False, "open_ports": [443],
              "unauthenticated_api": True, "encrypted_traffic": True}
    assert impl.solve(device) == "unauthenticated_api"


def test_example_unencrypted_traffic():
    device = {"default_password": False, "firmware_outdated": False, "open_ports": [443],
              "unauthenticated_api": False, "encrypted_traffic": False}
    assert impl.solve(device) == "unencrypted_traffic"


def test_example_secure():
    device = {"default_password": False, "firmware_outdated": False, "open_ports": [443],
              "unauthenticated_api": False, "encrypted_traffic": True}
    assert impl.solve(device) == "secure"


# ---- hidden / edge cases -------------------------------------------------
def test_edge_empty_dict_is_secure():
    # missing keys default to the safe value for that key
    assert impl.solve({}) == "secure"


def test_edge_empty_open_ports():
    device = {"default_password": False, "firmware_outdated": False, "open_ports": [],
              "unauthenticated_api": False, "encrypted_traffic": True}
    assert impl.solve(device) == "secure"


def test_port_21_also_exposed():
    device = {"default_password": False, "firmware_outdated": False, "open_ports": [21, 8080],
              "unauthenticated_api": False, "encrypted_traffic": True}
    assert impl.solve(device) == "exposed_port"


def test_other_ports_do_not_count():
    device = {"default_password": False, "firmware_outdated": False, "open_ports": [80, 8080, 3389],
              "unauthenticated_api": False, "encrypted_traffic": True}
    assert impl.solve(device) == "secure"


def test_priority_order():
    # everything is wrong at once — must return the HIGHEST priority verdict
    device = {"default_password": True, "firmware_outdated": True, "open_ports": [23, 21],
              "unauthenticated_api": True, "encrypted_traffic": False}
    assert impl.solve(device) == "default_credentials"

    # no default password, but everything else wrong -> outdated_firmware wins next
    device2 = {"default_password": False, "firmware_outdated": True, "open_ports": [23],
               "unauthenticated_api": True, "encrypted_traffic": False}
    assert impl.solve(device2) == "outdated_firmware"

    # only port + api + traffic wrong -> exposed_port wins over the two below it
    device3 = {"default_password": False, "firmware_outdated": False, "open_ports": [23],
               "unauthenticated_api": True, "encrypted_traffic": False}
    assert impl.solve(device3) == "exposed_port"


# ---- run every test above ------------------------------------------------
_results = []
for _name, _fn in list(globals().items()):
    if _name.startswith("test_") and callable(_fn):
        try:
            _fn()
            _results.append((_name, None))
        except Exception as _e:
            _results.append((_name, _e))
for _name, _e in _results:
    print(("PASS  " if _e is None else "FAIL  ") + _name + ("" if _e is None else f": {_e!r}"))
print(f"{sum(e is None for _, e in _results)}/{len(_results)} tests passed")
